In [53]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
# import matplotlib.pyplot as plt
# import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Research questions

- <mark>Can we identify distinct groups of artists who are enjoyed by similar listeners?</mark>

- Can we predict which songs a user is likely to add to their genre based on their listening history and similar users’ behavior?
  - Engineered fields: Song popularity (Number of times a song appears across all playlists) and Collaborative similarity index (Overlap in playlist content between users)
- Can we predict whether a song will be added to a playlist , based on the song elements (key, tempo, genre, and loudness) of the songs that already exist in the playlist?

In [4]:
df = pd.read_csv('../../data/clean_data.csv')
df.head()

,user,artist,playlist,tracks,spotify_popularity,genre,key,tempo,popularity,energy,...,good_for_work_study,good_for_relaxation_meditation,good_for_exercise,good_for_running,good_for_yoga_stretching,good_for_driving,good_for_social_gatherings,good_for_morning_routine,length_seconds,loudness_db_num
0,00055176fea33f6e027cd3302289378b,5 Seconds Of Summer,favs,10,0.025129,"pop,pop rock,pop punk",D Maj,134.162338,44.811688,80.545455,...,0.000000,0.000000,0.285714,0.162338,0.0,0.038961,0.006494,0.032468,210.675325,-4.913896
1,00055176fea33f6e027cd3302289378b,Against The Current,favs,3,0.001571,"acoustic,pop rock,rock",G Maj,121.264706,37.147059,84.441176,...,0.000000,0.000000,0.352941,0.088235,0.0,0.029412,0.000000,0.029412,199.794118,-4.070000
2,00055176fea33f6e027cd3302289378b,All Time Low,favs,8,0.030531,"alternative rock,emo,pop punk",D Maj,142.752066,38.801653,87.652893,...,0.008264,0.000000,0.206612,0.049587,0.0,0.000000,0.000000,0.000000,202.413223,-4.211901
3,00055176fea33f6e027cd3302289378b,Austin Mahone,favs,1,0.016145,pop,B Maj,111.692308,31.871795,63.461538,...,0.076923,0.051282,0.230769,0.025641,0.0,0.128205,0.000000,0.230769,198.230769,-5.960256
4,00055176fea33f6e027cd3302289378b,Avril Lavigne,favs,2,0.071806,"rock,pop,alternative rock",F Maj,128.726316,57.505263,79.252632,...,0.005263,0.000000,0.242105,0.152632,0.0,0.005263,0.010526,0.021053,214.731579,-4.756263


# Preprocessing

Let's cluster this data to find similar groups

In [5]:
# separate columns by dtypes
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Categorical columns: ', cat_cols)
print('Numerical columns: ', num_cols)
df.isnull().sum().sum()

Categorical columns:  ['user', 'artist', 'playlist', 'genre', 'key']
Numerical columns:  ['tracks', 'spotify_popularity', 'tempo', 'popularity', 'energy', 'danceability', 'positiveness', 'speechiness', 'liveness', 'acousticness', 'instrumentalness', 'good_for_party', 'good_for_work_study', 'good_for_relaxation_meditation', 'good_for_exercise', 'good_for_running', 'good_for_yoga_stretching', 'good_for_driving', 'good_for_social_gatherings', 'good_for_morning_routine', 'length_seconds', 'loudness_db_num']


0

In [6]:
# run tf_idf for clustering (exclude id columns like user/artist)
tf_idf_cols = ['playlist', 'genre', 'key']

# create vectorizers
playlist_vectorizer = TfidfVectorizer(
    max_df=0.5,
    min_df=5,
    stop_words="english",
)
playlist_tfidf = playlist_vectorizer.fit_transform(df['playlist'])

genre_vectorizer = TfidfVectorizer(
    max_df=0.5,
    min_df=5,
    stop_words="english",
)
genre_tfidf = genre_vectorizer.fit_transform(df['genre'])

key_vectorizer = TfidfVectorizer(
    max_df=0.5,
    min_df=5,
    stop_words="english",
)
key_tfidf = key_vectorizer.fit_transform(df['key'])

In [7]:
# use kmeans to cluster professions together
# tweaked n_clusters using elbow method below
playlist_kmeans = KMeans(n_clusters=35, random_state=42)
# fit the model
playlist_kmeans.fit(playlist_tfidf)
# store cluster labels in a variable
playlist_clusters = playlist_kmeans.labels_
df['playlist_cluster'] = playlist_clusters
df_playlist_cluster = df[['playlist', 'playlist_cluster']].drop_duplicates().copy()
# look at this to see if it makes sense
df_playlist_cluster

/Users/brandon/miniforge3/envs/mlenv_now/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


,playlist,playlist_cluster
0,favs,6
41,Fav songs,7
49,Sad songs,7
51,2014,29
83,Agnetha Fältskog,6
...,...,...
3026574,l a d y anthems,6
3026588,KickAss,6
3026645,natural elements,6
3026648,5:19,6


# Playlist clustering notes
## k=35

- 0: baby
- 1: starred
- 2: free
- 3: radio
- 4: 80's
- 5: good, covers, others (this may be misc)
- 6: misc
- 7: songs
- 8: list
- 9: rock
- 10: day
- 11: library
- 12: time
- 13: playlist
- 14: work
- 15: party
- 16: country & summer
- 17: pop
- 18: new
- 19: 2012
- 20: live
- 21: hip
- 22: various artists
- 23: music
- 24: warm
- 25: deluxe
- 26: 2015
- 27: favorites
- 28: dance
- 29: 2014
- 30: smooth
- 31: hits
- 32: stuff
- 33: 2013
- 34: hip hop

In [8]:
df_playlist_cluster['playlist_cluster'].value_counts()

6     105995
22      2375
25      1424
7       1419
23      1409
20      1286
13      1251
29      1220
9       1184
33      1081
18       934
19       852
16       767
17       745
26       677
28       652
12       598
10       573
5        563
15       553
31       503
34       438
32       394
8        392
14       231
0        223
4        212
27       211
2        115
30        76
11        44
3         37
24        35
21        29
1         10
Name: playlist_cluster, dtype: int64

In [9]:
# commenting out due to large runtime; cluster analysis recorded in notes
# # optimize clustering - playlist
# # find optimal k to use for grouping (hopefully get that 0 cluster down)
# inertia = []

# # we'll go up to 20 clusters
# upper = 40

# # loop through, try every k value from 2-20
# for k in range(2, upper):
#     kmeans = KMeans(n_clusters=k, random_state=0)
#     kmeans.fit(playlist_tfidf)
#     inertia.append(kmeans.inertia_)

In [10]:
# # elbow method to find optimal k
# fig = plt.figure(figsize=(10, 6)) 
# plt.plot(range(2, upper), inertia, marker='o')
# plt.title('Elbow Method for Optimal k')
# plt.xlabel('Number of Clusters (k)')
# plt.ylabel('Inertia (Sum of Squared Distances)')
# plt.xticks(range(2, upper))
# plt.show()

In [11]:
df.head()

,user,artist,playlist,tracks,spotify_popularity,genre,key,tempo,popularity,energy,...,good_for_relaxation_meditation,good_for_exercise,good_for_running,good_for_yoga_stretching,good_for_driving,good_for_social_gatherings,good_for_morning_routine,length_seconds,loudness_db_num,playlist_cluster
0,00055176fea33f6e027cd3302289378b,5 Seconds Of Summer,favs,10,0.025129,"pop,pop rock,pop punk",D Maj,134.162338,44.811688,80.545455,...,0.000000,0.285714,0.162338,0.0,0.038961,0.006494,0.032468,210.675325,-4.913896,6
1,00055176fea33f6e027cd3302289378b,Against The Current,favs,3,0.001571,"acoustic,pop rock,rock",G Maj,121.264706,37.147059,84.441176,...,0.000000,0.352941,0.088235,0.0,0.029412,0.000000,0.029412,199.794118,-4.070000,6
2,00055176fea33f6e027cd3302289378b,All Time Low,favs,8,0.030531,"alternative rock,emo,pop punk",D Maj,142.752066,38.801653,87.652893,...,0.000000,0.206612,0.049587,0.0,0.000000,0.000000,0.000000,202.413223,-4.211901,6
3,00055176fea33f6e027cd3302289378b,Austin Mahone,favs,1,0.016145,pop,B Maj,111.692308,31.871795,63.461538,...,0.051282,0.230769,0.025641,0.0,0.128205,0.000000,0.230769,198.230769,-5.960256,6
4,00055176fea33f6e027cd3302289378b,Avril Lavigne,favs,2,0.071806,"rock,pop,alternative rock",F Maj,128.726316,57.505263,79.252632,...,0.000000,0.242105,0.152632,0.0,0.005263,0.010526,0.021053,214.731579,-4.756263,6


In [12]:
# make sure you didn't do anything wrong
df.isnull().sum().sum()

0

In [13]:
# use kmeans to cluster genres together
genre_k = 12
# tweaked n_clusters using elbow method below
genre_kmeans = KMeans(n_clusters=genre_k, random_state=42)
# fit the model
genre_kmeans.fit(genre_tfidf)
# store cluster labels in a variable
genre_clusters = genre_kmeans.labels_
df['genre_cluster'] = genre_clusters
df_genre_cluster = df[['genre', 'genre_cluster']].drop_duplicates().copy()
# look at this to see if it makes sense
df_genre_cluster

/Users/brandon/miniforge3/envs/mlenv_now/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


,genre,genre_cluster
0,"pop,pop rock,pop punk",5
1,"acoustic,pop rock,rock",0
2,"alternative rock,emo,pop punk",9
3,pop,5
4,"rock,pop,alternative rock",0
...,...,...
2919402,"progressive rock,metal,progressive metal",9
2931633,"j-pop,indie,pop",8
2960338,"punk,pop rock",5
2973903,"rock,hip-hop,reggae",1


# Genre clustering notes

## k = 12

- 0: rock
- 1: hip-hop
- 2: electronic
- 3: country
- 4: soul
- 5: pop
- 6: synth
- 7: folk
- 8: indie
- 9: misc / rock / metal / punk
- 10: electropop
- 11: jazz

In [14]:
df_genre_cluster['genre_cluster'].value_counts()

9     967
0     359
2     240
8     215
11    129
7     119
4     115
1     101
5      91
6      82
10     81
3      80
Name: genre_cluster, dtype: int64

In [15]:
# # commenting out due to large runtime; cluster analysis recorded in notes
# # optimize clustering - playlist
# # find optimal k to use for grouping (hopefully get that 0 cluster down)
# inertia = []

# # we'll go up to 20 clusters
# upper = 20

# # loop through, try every k value from 2-20
# for k in range(2, upper):
#     kmeans = KMeans(n_clusters=k, random_state=0)
#     kmeans.fit(genre_tfidf)
#     inertia.append(kmeans.inertia_)

In [16]:
# # elbow method to find optimal k
# fig = plt.figure(figsize=(10, 6)) 
# plt.plot(range(2, upper), inertia, marker='o')
# plt.title('Elbow Method for Optimal k')
# plt.xlabel('Number of Clusters (k)')
# plt.ylabel('Inertia (Sum of Squared Distances)')
# plt.xticks(range(2, upper))
# plt.show()

In [17]:
# see if key needs to be clustered or one-hot, etc
# with 24 items, go with label encoding or target encoding (below)
df['key'].drop_duplicates()

0       D Maj
1       G Maj
3       B Maj
4       F Maj
5      F# Maj
6       C Maj
8      C# min
12     D# Maj
13     G# Maj
14     D# min
17      A Maj
19      F min
21     A# Maj
23      E min
30      E Maj
32      A min
33      B min
34     C# Maj
37      G min
43     F# min
44      D min
45     G# min
58     A# min
176     C min
Name: key, dtype: object

# Preprocess data for model

In [18]:
# TODO: finish preprocessing data and then train a model
df.head()

,user,artist,playlist,tracks,spotify_popularity,genre,key,tempo,popularity,energy,...,good_for_exercise,good_for_running,good_for_yoga_stretching,good_for_driving,good_for_social_gatherings,good_for_morning_routine,length_seconds,loudness_db_num,playlist_cluster,genre_cluster
0,00055176fea33f6e027cd3302289378b,5 Seconds Of Summer,favs,10,0.025129,"pop,pop rock,pop punk",D Maj,134.162338,44.811688,80.545455,...,0.285714,0.162338,0.0,0.038961,0.006494,0.032468,210.675325,-4.913896,6,5
1,00055176fea33f6e027cd3302289378b,Against The Current,favs,3,0.001571,"acoustic,pop rock,rock",G Maj,121.264706,37.147059,84.441176,...,0.352941,0.088235,0.0,0.029412,0.000000,0.029412,199.794118,-4.070000,6,0
2,00055176fea33f6e027cd3302289378b,All Time Low,favs,8,0.030531,"alternative rock,emo,pop punk",D Maj,142.752066,38.801653,87.652893,...,0.206612,0.049587,0.0,0.000000,0.000000,0.000000,202.413223,-4.211901,6,9
3,00055176fea33f6e027cd3302289378b,Austin Mahone,favs,1,0.016145,pop,B Maj,111.692308,31.871795,63.461538,...,0.230769,0.025641,0.0,0.128205,0.000000,0.230769,198.230769,-5.960256,6,5
4,00055176fea33f6e027cd3302289378b,Avril Lavigne,favs,2,0.071806,"rock,pop,alternative rock",F Maj,128.726316,57.505263,79.252632,...,0.242105,0.152632,0.0,0.005263,0.010526,0.021053,214.731579,-4.756263,6,0


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3027724 entries, 0 to 3027723
Data columns (total 29 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   user                            object 
 1   artist                          object 
 2   playlist                        object 
 3   tracks                          int64  
 4   spotify_popularity              float64
 5   genre                           object 
 6   key                             object 
 7   tempo                           float64
 8   popularity                      float64
 9   energy                          float64
 10  danceability                    float64
 11  positiveness                    float64
 12  speechiness                     float64
 13  liveness                        float64
 14  acousticness                    float64
 15  instrumentalness                float64
 16  good_for_party                  float64
 17  good_for_work_study        

In [20]:
# TODO: make categorical cluster dictionary from markdown notes
genre_cluster_dict = {
    0: 'rock',
    1: 'hip-hop',
    2: 'electronic',
    3: 'country',
    4: 'soul',
    5: 'pop',
    6: 'synth',
    7: 'folk',
    8: 'indie',
    9: 'misc',
    10: 'electropop',
    11: 'jazz'
}

playlist_cluster_dict = {
    0: 'baby',
    1: 'starred',
    2: 'free',
    3: 'radio',
    4: "80's",
    5: 'good, covers',
    6: 'misc',
    7: 'songs',
    8: 'list',
    9: 'rock',
    10: 'day',
    11: 'library',
    12: 'time',
    13: 'playlist',
    14: 'work',
    15: 'party',
    16: 'country & summer',
    17: 'pop',
    18: 'new',
    19: '2012',
    20: 'live',
    21: 'hip',
    22: 'various artists',
    23: 'music',
    24: 'warm',
    25: 'deluxe',
    26: '2015',
    27: 'favorites',
    28: 'dance',
    29: '2014',
    30: 'smooth',
    31: 'hits',
    32: 'stuff',
    33: '2013',
    34: 'hip hop'
}

# drop categoricals that have already been converted 
df = df.drop(['playlist', 'genre'], axis=1)

In [21]:
# convert key to numerical and keep dictionary
def label_encode(df, column):
    # encodes with label encoding, creates a dict, drops orig field
    le = LabelEncoder()
    encode_col = f'encoded_{column}'
    df[encode_col] = le.fit_transform(df[column])
    encode_dict = dict(zip(df[encode_col], df[column]))
    # drop key column
    df = df.drop(column, axis=1)
    return df, encode_dict
df, key_dict = label_encode(df, 'key')

In [22]:
# handle user and artist columns
df, user_dict = label_encode(df, 'user')
df, artist_dict = label_encode(df, 'artist')

In [23]:
# scale columns
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

# Print the first few rows of the scaled DataFrame
df_scaled.head()

,tracks,spotify_popularity,tempo,popularity,energy,danceability,positiveness,speechiness,liveness,acousticness,...,good_for_driving,good_for_social_gatherings,good_for_morning_routine,length_seconds,loudness_db_num,playlist_cluster,genre_cluster,encoded_key,encoded_user,encoded_artist
0,0.671333,-0.509285,1.001246,0.738532,0.965072,-0.249899,-0.203578,0.595331,1.088437,-0.882902,...,-0.077897,-0.148138,-0.304741,-0.507071,1.075488,-0.324883,0.163329,0.054619,-1.72247,-1.727603
1,-0.007288,-0.920003,0.011918,0.126424,1.199092,0.186841,0.487413,-0.344771,-0.427640,-0.937822,...,-0.176448,-0.256234,-0.328255,-0.701726,1.364636,-0.324883,-1.179070,1.541367,-1.72247,-1.674383
2,0.477441,-0.415093,1.660131,0.258562,1.392023,-0.706057,0.245248,0.045197,0.088073,-0.961177,...,-0.479986,-0.256234,-0.554577,-0.654872,1.316016,-0.324883,1.237248,0.054619,-1.72247,-1.634536
3,-0.201179,-0.665905,-0.722344,-0.294866,-0.061175,0.943107,0.442049,-0.275094,-0.464832,-0.296492,...,0.843129,-0.256234,1.221178,-0.729692,0.716968,-0.324883,0.163329,-0.837430,-1.72247,-1.523320
4,-0.104234,0.304484,0.584270,1.752258,0.887411,-0.339330,-0.071477,-0.276588,0.473270,-0.825661,...,-0.425669,-0.081005,-0.392578,-0.434508,1.129499,-0.324883,-1.179070,0.946668,-1.72247,-1.517452


In [24]:
# create a k-means model and explore w/ parallel coordinates graph
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_scaled['cluster'] = kmeans.fit_predict(df_scaled)

In [25]:
# need to make a subset of data to avoid memory issues
sample_size = 500000
df_sample = df_scaled.sample(sample_size)

In [26]:
df_sample.columns

Index(['tracks', 'spotify_popularity', 'tempo', 'popularity', 'energy',
       'danceability', 'positiveness', 'speechiness', 'liveness',
       'acousticness', 'instrumentalness', 'good_for_party',
       'good_for_work_study', 'good_for_relaxation_meditation',
       'good_for_exercise', 'good_for_running', 'good_for_yoga_stretching',
       'good_for_driving', 'good_for_social_gatherings',
       'good_for_morning_routine', 'length_seconds', 'loudness_db_num',
       'playlist_cluster', 'genre_cluster', 'encoded_key', 'encoded_user',
       'encoded_artist', 'cluster'],
      dtype='object')

In [27]:
# Create a parallel coordinates graph for visualization
# commented out to keep notebook size down
# subset_cols = ['cluster', 'playlist_cluster', 'genre_cluster',
#                'encoded_key', 'encoded_user', 'encoded_artist',
#               'tempo', 'tracks', 'positiveness']
# fig = px.parallel_coordinates(df_sample[subset_cols], color="cluster")
# fig.show()

In [28]:
# run summary stats on each cluster and feature, look at medians
df_scaled.groupby('cluster').median()

,tracks,spotify_popularity,tempo,popularity,energy,danceability,positiveness,speechiness,liveness,acousticness,...,good_for_driving,good_for_social_gatherings,good_for_morning_routine,length_seconds,loudness_db_num,playlist_cluster,genre_cluster,encoded_key,encoded_user,encoded_artist
cluster,,,,,,,,,,,,,,,,,,,,,
0,-0.201179,-0.488475,-0.444217,-0.112804,-0.125307,0.859272,0.605922,-0.086698,-0.178337,-0.186660,...,0.313883,-0.256234,0.158861,-0.130771,0.018081,-0.324883,-0.105151,-0.24273,-0.004027,-0.037255
1,-0.201179,-0.515856,-0.628029,-0.349131,-1.567313,-0.429312,-0.739768,-0.531119,-0.516922,1.743124,...,-0.479986,-0.256234,-0.554577,-0.218019,-1.315911,-0.324883,0.431809,-0.24273,-0.013631,-0.129776
2,-0.201179,-0.355950,0.201277,-0.204766,0.331597,-0.615180,-0.386816,-0.410818,-0.007099,-0.408614,...,-0.479986,-0.256234,-0.384835,0.043941,0.296186,-0.324883,-0.373630,-0.24273,-0.022789,0.195138
3,-0.201179,-0.009852,0.411479,0.502578,0.653333,0.562951,0.464125,-0.222347,-0.098406,-0.693772,...,-0.479986,-0.256234,-0.244713,-0.334744,0.736287,-0.324883,-0.105151,-0.24273,-0.025022,-0.137282


In [ ]:
# add clusters to original df for analysis/visualization
df['cluster'] = df_scaled['cluster']
df.head()

In [51]:
# TODO: make radar charts
focus_cols = ['popularity', 'energy', 'danceability', 'positiveness', 'speechiness']
df_grouped_med = df.groupby('cluster').median()
df_grouped_med.head()
df_graph = df_grouped_med[focus_cols].copy()
df_graph.head(1)

,popularity,energy,danceability,positiveness,speechiness
cluster,,,,,
0,34.151515,62.393939,66.402985,59.111111,7.02069


In [61]:
# play around with indexing to get data format right for plot
df_graph.iloc[0]
df_graph.iloc[0].index
df_graph.iloc[0].values

categories = df_graph.iloc[0].index

fig = go.Figure()

# cluster 0
fig.add_trace(go.Scatterpolar(
      r=df_graph.iloc[0].values,
      theta=categories,
      # fill='toself',
      name='Cluster 0'
))

fig.add_trace(go.Scatterpolar(
      r=df_graph.iloc[1].values,
      theta=categories,
      # fill='toself',
      name='Cluster 1'
))

fig.add_trace(go.Scatterpolar(
      r=df_graph.iloc[2].values,
      theta=categories,
      # fill='toself',
      name='Cluster 2'
))

fig.add_trace(go.Scatterpolar(
      r=df_graph.iloc[3].values,
      theta=categories,
      # fill='toself',
      name='Cluster 3'
))

fig.update_layout(
  polar=dict(
    radialaxis=dict(
      visible=True,
      range=[0, 100]
    )),
  showlegend=True
)

In [ ]:


categories = ['processing cost','mechanical properties','chemical stability',
              'thermal stability', 'device integration']

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
      r=[1, 5, 2, 2, 3],
      theta=categories,
      fill='toself',
      name='Product A'
))
fig.add_trace(go.Scatterpolar(
      r=[4, 3, 2.5, 1, 2],
      theta=categories,
      fill='toself',
      name='Product B'
))

fig.update_layout(
  polar=dict(
    radialaxis=dict(
      visible=True,
      range=[0, 5]
    )),
  showlegend=False
)

fig.show()

In [ ]:
# # TODO: try gaussian mixture model if you have time (from lecture)
# # Fit a Gaussian Mixture Model
# gmm = GaussianMixture(n_components = 3, covariance_type = 'full', random_state = 0)
# gmm.fit (mlb_preprocessed)
# # Predict soft assignments
# probs = gmm.predict_proba(mlb_preprocessed)
# # Assign each data point to the cluster with the highest probability
# hard_assignments = np.argmax (probs, axis=1)
# # Calculate the silhouette score using these hard assignments
# silhouette_avg = silhouette_score(mlb_preprocessed, hard_assignments)
# print (f"The average silhouette score for the GMM clustering is: {silhouette_avg: .3f}")
# # BIC
# bic_score = gmm.bic(mlb_preprocessed)
# print(f"BIC Score: {bic_score:.3f}")
# # Convergence
# print (f"Converged: {gmm. converged_}")